<center><h1>Naive and Basic Baseline Modeling</center></h1>

### Import Packages and Load Data

In [1]:
import pandas as pd
import duckdb

from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression


In [2]:
df_logistic = pd.read_csv("df_ready.csv")

### Naive Baseline Model
Using the winrate of the previous split, the baseline model predicts the results of the next split.

In [ ]:
# Connect to saved database file
con = duckdb.connect("lol_predictors.duckdb")

baseline_query_2014 = con.execute("""
    SELECT 
        teamname, 
        CAST(SUM(result) AS DOUBLE) / COUNT(*) as win_rate,
        COUNT(*) as games_played
    FROM df_2014
    WHERE position = 'team' AND split = 'Spring'
    GROUP BY teamname
""").df()

test_matches = con.execute("""
    SELECT 
        teamname,
        result
    FROM df_2014
    WHERE position = 'team' AND split = 'Summer'
""").df()

con.close()

In [16]:
merged = test_matches.merge(baseline_query_2014, on='teamname', how='left')
merged['win_rate'] = merged['win_rate'].fillna(0.5)

In [17]:
merged['prediction'] = (merged['win_rate'] >= 0.5).astype(int)

In [ ]:
acc = accuracy_score(merged['result'], merged['prediction'])
print(f"Baseline Accuracy: {acc * 100:.2f}%")

### Logistic Regression Model